# Кольцевые коммуникации

Реализуйте три операции на CPU с Gloo. 
- `rank` — номер процесса,
- `world_size` — число процессов.
- Входные тензоры — одномерные CPU-тензоры одинакового размера на всех процессах. Их менять нельзя

In [1]:
%%writefile collectives.py
"""Ring collectives implemented with point-to-point torch.distributed calls."""

import torch
import torch.distributed as dist


def _ring():
    if not dist.is_available() or not dist.is_initialized():
        raise RuntimeError("torch.distributed process group is not initialized")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    return rank, world_size, (rank - 1) % world_size, (rank + 1) % world_size


def _check_vector(tensor, name):
    if tensor.ndim != 1:
        raise ValueError(f"{name} must be a one-dimensional tensor")
    if tensor.device.type != "cpu":
        raise ValueError(f"{name} must be a CPU tensor for the Gloo exercise")


def ring_reduce_scatter(tensor):
    _check_vector(tensor, "tensor")
    rank, world_size, left, right = _ring()
    if tensor.numel() % world_size:
        raise ValueError("tensor length must be divisible by world_size")
    chunks = [chunk.clone() for chunk in tensor.chunk(world_size)]
    if world_size == 1:
        return chunks[0]

    receive_buffer = torch.empty_like(chunks[0])
    for step in range(world_size - 1):
        send_index = (rank - step - 1) % world_size
        receive_index = (rank - step - 2) % world_size
        receive_request = dist.irecv(receive_buffer, src=left)
        send_request = dist.isend(chunks[send_index], dst=right)
        receive_request.wait()
        send_request.wait()
        chunks[receive_index].add_(receive_buffer)
    return chunks[rank]


def ring_all_gather(shard):
    _check_vector(shard, "shard")
    rank, world_size, left, right = _ring()
    chunks = [torch.empty_like(shard) for _ in range(world_size)]
    chunks[rank].copy_(shard)
    for step in range(world_size - 1):
        send_index = (rank - step) % world_size
        receive_index = (rank - step - 1) % world_size
        receive_request = dist.irecv(chunks[receive_index], src=left)
        send_request = dist.isend(chunks[send_index], dst=right)
        receive_request.wait()
        send_request.wait()
    return torch.cat(chunks)


def ring_all_reduce(tensor):
    return ring_all_gather(ring_reduce_scatter(tensor))

Overwriting collectives.py


## Сравнение с PyTorch

Сравним нашу реализацию с готовыми функциями торча `dist.reduce_scatter_tensor`, `dist.all_gather`
и `dist.all_reduce` на нескольких размерах тензоров. 

In [2]:
%%writefile check.py
from datetime import timedelta

import torch
import torch.distributed as dist

from collectives import ring_all_gather, ring_all_reduce, ring_reduce_scatter


torch.set_num_threads(1)
dist.init_process_group("gloo", timeout=timedelta(seconds=30))
try:
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    generator = torch.Generator().manual_seed(42 + rank)

    for chunk_size in (1, 7, 31):
        values = torch.randn(world_size * chunk_size, generator=generator, dtype=torch.float64)
        original = values.clone()
        expected_shard = torch.empty(chunk_size, dtype=values.dtype)
        dist.reduce_scatter_tensor(expected_shard, values.clone(), op=dist.ReduceOp.SUM)
        torch.testing.assert_close(ring_reduce_scatter(values), expected_shard)
        torch.testing.assert_close(values, original, rtol=0, atol=0)

        shard = torch.randn(chunk_size, generator=generator, dtype=torch.float64)
        original_shard = shard.clone()
        expected_parts = [torch.empty_like(shard) for _ in range(world_size)]
        dist.all_gather(expected_parts, shard)
        torch.testing.assert_close(ring_all_gather(shard), torch.cat(expected_parts))
        torch.testing.assert_close(shard, original_shard, rtol=0, atol=0)

        expected_sum = values.clone()
        dist.all_reduce(expected_sum, op=dist.ReduceOp.SUM)
        torch.testing.assert_close(ring_all_reduce(values), expected_sum)
        torch.testing.assert_close(values, original, rtol=0, atol=0)

    if rank == 0:
        print(f"OK: все три операции прошли проверки на {world_size} процессах.")
finally:
    dist.destroy_process_group()

Overwriting check.py


In [3]:
import sys

!{sys.executable} -m torch.distributed.run --nnodes=1 --node-rank=0 --master-addr=127.0.0.1 --master-port=29501 --nproc-per-node=2 check.py

W1008 17:46:27.037679 30404 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.



*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************


[W1008 17:46:28.318713000 ProcessGroupGloo.cpp:545] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
[W1008 17:46:28.320679000 ProcessGroupGloo.cpp:545] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())


OK: все три операции прошли проверки на 2 процессах.


In [4]:
!{sys.executable} -m torch.distributed.run --nnodes=1 --node-rank=0 --master-addr=127.0.0.1 --master-port=29502 --nproc-per-node=4 check.py

W1008 17:46:29.771630 30444 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************


[W1008 17:46:30.823765000 ProcessGroupGloo.cpp:545] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
[W1008 17:46:30.826247000 ProcessGroupGloo.cpp:545] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
[W1008 17:46:30.828456000 ProcessGroupGloo.cpp:545] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
[W1008 17:46:30.828478000 ProcessGroupGloo.cpp:545] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())


OK: все три операции прошли проверки на 4 процессах.
